# Warehouse Operations Database

## Introduction

This notebook guides you through exploring a warehouse distribution database and implementing functions that analyze its data using both SQL and Python.

Work through the notebook from top to bottom. The first section explores the database structure. The second section provides two complete examples. The third section contains your tasks - implement the functions in the cells marked **YOUR CODE HERE**, then run the test cells that follow to verify your work.

## Part 1: Explore the Database

In [ ]:
import sqlite3
import os

# If running on Colab, download the database file
if not os.path.exists('warehouse.db'):
    import urllib.request
    url = "https://raw.githubusercontent.com/olearydj/INSY3010/main/warehouse.db"
    urllib.request.urlretrieve(url, "warehouse.db")
    print("Downloaded warehouse.db")

# Connect to the warehouse database
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

# Verify the connection
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [t[0] for t in cursor.fetchall()]

if 'Warehouse' not in tables:
    print("ERROR: Database appears empty or corrupt!")
else:
    print("Connected to warehouse.db")
    print(f"Tables: {tables}")

### Warehouse Table

| Column | Description |
|--------|-------------|
| `warehouse_id` | Unique identifier (e.g., "W001") |
| `name` | Distribution center name |
| `city`, `state` | Location |
| `capacity_sqft` | Storage capacity in square feet |

In [ ]:
cursor.execute("SELECT * FROM Warehouse")
warehouses = cursor.fetchall()

print("Distribution Centers:")
for row in warehouses:
    print(f"  {row[0]}: {row[1]} ({row[2]}, {row[3]}) - {row[4]:,} sq ft")

### Product Table

| Column | Description |
|--------|-------------|
| `product_id` | Unique identifier (e.g., "P0001") |
| `product_name` | Name of the product |
| `category` | Electronics, Hardware, Safety, or Packaging |
| `unit_weight` | Weight per unit in pounds |
| `unit_cost` | Cost per unit in dollars |

In [ ]:
# Products by category
cursor.execute("SELECT category, COUNT(*) FROM Product GROUP BY category")
print("Products by Category:")
for cat, count in cursor.fetchall():
    print(f"  {cat}: {count} products")

# Sample products
cursor.execute("SELECT * FROM Product LIMIT 6")
print("\nSample Products:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} ({row[2]}) - {row[3]} lbs, ${row[4]:.2f}")

### Carrier Table

| Column | Description |
|--------|-------------|
| `carrier_id` | Unique identifier (e.g., "C001") |
| `name` | Carrier company name |
| `mode` | Shipping mode: Ground, Air, or Rail |
| `base_rate` | Rate per pound-unit in dollars |

In [ ]:
cursor.execute("SELECT * FROM Carrier ORDER BY mode, name")
print("Carriers:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} ({row[2]}) - ${row[3]:.2f}/lb-unit")

### Shipment Table

The Shipment table records each shipment event. It connects Warehouse, Product, and Carrier with foreign keys to all three.

| Column | Description |
|--------|-------------|
| `shipment_id` | Unique identifier |
| `warehouse_id` | FK to Warehouse |
| `product_id` | FK to Product |
| `carrier_id` | FK to Carrier |
| `ship_date` | Date of shipment (YYYY-MM-DD) |
| `quantity` | Number of units shipped |
| `shipment_type` | Inbound or Outbound |
| `status` | Delivered, In Transit, or Delayed |

In [ ]:
# Sample shipments with names from all three related tables
cursor.execute("""
    SELECT s.shipment_id, w.name, p.product_name, c.name,
           s.quantity, s.shipment_type, s.status
    FROM Shipment s
    JOIN Warehouse w ON s.warehouse_id = w.warehouse_id
    JOIN Product p ON s.product_id = p.product_id
    JOIN Carrier c ON s.carrier_id = c.carrier_id
    LIMIT 5
""")

print("Sample Shipments:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[5]} - {row[4]} units of {row[2]}")
    print(f"           {row[1]} via {row[3]} [{row[6]}]")

### Exploring the Data

In [ ]:
# How many different products does each warehouse handle?
cursor.execute("""
    SELECT w.name, COUNT(DISTINCT s.product_id) AS product_count
    FROM Warehouse w
    JOIN Shipment s ON w.warehouse_id = s.warehouse_id
    GROUP BY w.warehouse_id, w.name
    ORDER BY product_count DESC
""")

print("Product Variety by Warehouse:")
for name, count in cursor.fetchall():
    print(f"  {name}: {count} different products")

In [ ]:
# Shipment volume: inbound vs outbound
cursor.execute("""
    SELECT shipment_type, COUNT(*) AS num_shipments, SUM(quantity) AS total_units
    FROM Shipment
    GROUP BY shipment_type
""")

print("Shipment Volume:")
for stype, count, total in cursor.fetchall():
    print(f"  {stype}: {count} shipments, {total:,} total units")

In [ ]:
# Shipment status overview
cursor.execute("""
    SELECT status, COUNT(*) AS num_shipments
    FROM Shipment
    GROUP BY status
    ORDER BY num_shipments DESC
""")

print("Shipment Status:")
for status, count in cursor.fetchall():
    print(f"  {status}: {count} shipments")

---

## Part 2: Examples

Before starting the tasks, study these two examples. Both answer a question about the data, but they use different approaches.

- Example 1 uses **SQL** to do all the filtering and sorting
- Example 2 uses **Python** to filter and sort after fetching all the data

Understanding the difference is key to the tasks that follow.

### Example 1: SQL Approach

Find all products in a specific category. SQL handles the filtering and sorting.

In [ ]:
def get_products_by_category(conn, category):
    """
    Find all products in a specific category using SQL.

    Args:
        conn: Database connection object
        category (str): Product category (e.g., "Electronics", "Hardware")

    Returns:
        list of tuples: (product_id, product_name, unit_weight, unit_cost)
    """
    cursor = conn.cursor()
    query = """
        SELECT product_id, product_name, unit_weight, unit_cost
        FROM Product
        WHERE category = ?
        ORDER BY product_name
    """
    cursor.execute(query, (category,))
    return cursor.fetchall()


# Test it
results = get_products_by_category(conn, "Electronics")
print(f"Found {len(results)} electronics products:")
for pid, name, weight, cost in results:
    print(f"  {pid}: {name} - {weight} lbs, ${cost:.2f}")

### Example 2: Python Approach

Find products heavier than a given weight. SQL fetches all products; Python does the filtering and sorting.

In [ ]:
def get_heavy_products(conn, min_weight):
    """
    Find products that weigh more than a specified amount using Python.

    Args:
        conn: Database connection object
        min_weight (float): Minimum unit weight in pounds

    Returns:
        list of tuples: (product_name, category, unit_weight) sorted by
                        weight descending
    """
    cursor = conn.cursor()
    # Get ALL products - no filtering in SQL
    cursor.execute("SELECT product_name, category, unit_weight FROM Product")
    all_products = cursor.fetchall()

    # Filter using Python
    heavy = []
    for name, category, weight in all_products:
        if weight > min_weight:
            heavy.append((name, category, weight))

    # Sort by weight descending using Python
    heavy.sort(key=lambda x: x[2], reverse=True)
    return heavy


# Test it
results = get_heavy_products(conn, 10.0)
print(f"Found {len(results)} products over 10 lbs:")
for name, category, weight in results:
    print(f"  {name} ({category}) - {weight} lbs")

---

## Part 3: Your Tasks

For each task below:
1. Read the provided implementation to understand what it does
2. Implement the missing function in the **YOUR CODE HERE** cell
3. Run the test cell that follows to verify your work

### Task 1: Implement the SQL Version

The Python implementation below calculates total quantity handled by each warehouse. It fetches raw data, groups it with a dictionary, and sorts the result.

Your job: implement `get_warehouse_totals_sql()` to produce the same result using SQL.

In [ ]:
# PROVIDED - Python implementation (read and understand this)

def get_warehouse_totals_python(conn):
    """
    Calculate total quantity handled by each warehouse - PYTHON VERSION.

    Returns:
        list of tuples: (warehouse_name, total_quantity) ordered by
                        total_quantity DESC
    """
    cursor = conn.cursor()

    # Get shipment data with warehouse names
    cursor.execute("""
        SELECT w.name, s.quantity
        FROM Warehouse w
        JOIN Shipment s ON w.warehouse_id = s.warehouse_id
    """)
    data = cursor.fetchall()

    # Group by warehouse in Python
    warehouse_totals = {}
    for warehouse_name, quantity in data:
        if warehouse_name not in warehouse_totals:
            warehouse_totals[warehouse_name] = 0
        warehouse_totals[warehouse_name] += quantity

    # Convert to list of tuples
    results = [(name, total) for name, total in warehouse_totals.items()]

    # Sort by total descending
    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [ ]:
# YOUR CODE HERE - Implement the SQL version

def get_warehouse_totals_sql(conn):
    """
    Calculate total quantity handled by each warehouse - SQL VERSION.

    Your SQL query should:
    1. JOIN Warehouse and Shipment tables
    2. GROUP BY warehouse to calculate totals
    3. Use SUM to add up quantity for each warehouse
    4. ORDER BY total quantity descending

    Returns:
        list of tuples: (warehouse_name, total_quantity) ordered by
                        total_quantity DESC

    Hint: Your query should look something like:
        SELECT w.name, SUM(...) AS total_quantity
        FROM ...
        JOIN ...
        GROUP BY ...
        ORDER BY ...
    """
    pass

In [ ]:
# TEST - Task 1 (run this cell to check your work)

python_result = get_warehouse_totals_python(conn)
sql_result = get_warehouse_totals_sql(conn)

if sql_result is None:
    print("Task 1: Not yet implemented (function returned None)")
else:
    try:
        assert len(sql_result) == len(python_result), \
            (f"SQL returned {len(sql_result)} warehouses "
             f"but Python returned {len(python_result)}")
        for i in range(len(sql_result)):
            sql_name, sql_total = sql_result[i]
            py_name, py_total = python_result[i]
            assert sql_name == py_name, \
                f"Warehouse mismatch at position {i}: SQL={sql_name}, Python={py_name}"
            assert sql_total == py_total, \
                f"Total mismatch for {sql_name}: SQL={sql_total}, Python={py_total}"

        print(f"TASK 1 PASSED!")
        print(f"  Both implementations found {len(sql_result)} warehouses")
        print(f"  Highest volume: {sql_result[0][0]} with {sql_result[0][1]:,} units")
    except AssertionError as e:
        print(f"TASK 1 FAILED: {e}")

### Task 2: Implement the Python Version

The SQL implementation below finds delayed shipments grouped by carrier. It filters, groups, counts, and sorts entirely in SQL.

Your job: implement `get_delayed_by_carrier_python()` to produce the same result using Python for the filtering, grouping, and sorting.

In [ ]:
# PROVIDED - SQL implementation (read and understand this)

def get_delayed_by_carrier_sql(conn):
    """
    Find delayed shipment counts and quantities by carrier - SQL VERSION.

    Returns:
        list of tuples: (carrier_name, delayed_count, delayed_quantity)
                        ordered by delayed_count DESC, then carrier name ASC
    """
    cursor = conn.cursor()
    query = """
        SELECT c.name, COUNT(*) AS delayed_count,
               SUM(s.quantity) AS delayed_quantity
        FROM Carrier c
        JOIN Shipment s ON c.carrier_id = s.carrier_id
        WHERE s.status = 'Delayed'
        GROUP BY c.carrier_id, c.name
        ORDER BY delayed_count DESC, c.name ASC
    """
    cursor.execute(query)
    return cursor.fetchall()

In [ ]:
# YOUR CODE HERE - Implement the Python version

def get_delayed_by_carrier_python(conn):
    """
    Find delayed shipment counts and quantities by carrier - PYTHON VERSION.

    Your implementation should:
    1. Fetch ALL shipment data with carrier names (use a JOIN to get carrier
       names, but do NOT filter in SQL - fetch everything)
    2. Filter for delayed shipments using Python (if statement)
    3. Group by carrier name using a dictionary, tracking count and quantity
    4. Convert to list of tuples
    5. Sort by count descending, then by carrier name ascending (to break
       ties consistently)

    Returns:
        list of tuples: (carrier_name, delayed_count, delayed_quantity)
                        ordered by delayed_count DESC, then carrier name ASC

    Hint: You might use a dictionary like:
        carrier_stats = {}
        for carrier_name, quantity, status in data:
            if status == 'Delayed':
                if carrier_name not in carrier_stats:
                    carrier_stats[carrier_name] = {'count': 0, 'quantity': 0}
                carrier_stats[carrier_name]['count'] += 1
                carrier_stats[carrier_name]['quantity'] += quantity
    """
    pass

In [ ]:
# TEST - Task 2 (run this cell to check your work)

sql_result = get_delayed_by_carrier_sql(conn)
python_result = get_delayed_by_carrier_python(conn)

if python_result is None:
    print("Task 2: Not yet implemented (function returned None)")
else:
    try:
        assert len(python_result) == len(sql_result), \
            (f"Python returned {len(python_result)} carriers "
             f"but SQL returned {len(sql_result)}")
        for i in range(len(sql_result)):
            sql_name, sql_count, sql_qty = sql_result[i]
            py_name, py_count, py_qty = python_result[i]
            assert sql_name == py_name, \
                f"Carrier mismatch at position {i}: SQL={sql_name}, Python={py_name}"
            assert sql_count == py_count, \
                f"Count mismatch for {sql_name}: SQL={sql_count}, Python={py_count}"
            assert sql_qty == py_qty, \
                f"Quantity mismatch for {sql_name}: SQL={sql_qty}, Python={py_qty}"

        print(f"TASK 2 PASSED!")
        print(f"  Both implementations found {len(python_result)} carriers with delays")
        print(f"  Most delays: {python_result[0][0]} "
              f"with {python_result[0][1]} delayed shipments")
    except AssertionError as e:
        print(f"TASK 2 FAILED: {e}")

### Task 3: Write Results to CSV

Implement a function that writes query results to a CSV (comma-separated values) file. This exercises file I/O: opening files safely with context managers, writing formatted output, and handling errors with `try/except`.

In [ ]:
# YOUR CODE HERE - Implement export_to_csv

def export_to_csv(data, headers, filename):
    """
    Write a list of data rows to a CSV file with headers.

    This function takes query results (a list of tuples) and column headers
    (a list of strings) and writes them to a CSV file. Each value should be
    separated by commas, with one row per line.

    Your implementation should:
    1. Use a context manager (with statement) to open the file for writing
    2. Write the header row first (comma-separated)
    3. Write each data row (comma-separated values)
    4. Use try/except to handle potential errors (e.g., invalid filename,
       permission denied) - print a useful error message if writing fails
    5. Return True if the file was written successfully, False otherwise

    Args:
        data (list of tuples): The rows to write
        headers (list of str): Column names for the header row
        filename (str): Path to the output file

    Returns:
        bool: True if file was written successfully, False otherwise

    Example:
        data = [("Alice", 95), ("Bob", 87)]
        headers = ["Name", "Score"]
        export_to_csv(data, headers, "scores.csv")

        # scores.csv would contain:
        # Name,Score
        # Alice,95
        # Bob,87
    """
    pass

In [ ]:
# TEST - Task 3 (run this cell to check your work)

test_data = get_warehouse_totals_python(conn)
test_headers = ["Warehouse", "Total Quantity"]
test_file = "warehouse_totals.csv"

result = export_to_csv(test_data, test_headers, test_file)

if result is None:
    print("Task 3: Not yet implemented (function returned None)")
elif result:
    # Verify the file was created and has correct content
    with open(test_file, "r") as f:
        lines = f.readlines()

    try:
        assert len(lines) == len(test_data) + 1, \
            (f"CSV has {len(lines)} lines but expected "
             f"{len(test_data) + 1} (header + data)")

        header_line = lines[0].strip()
        assert header_line == "Warehouse,Total Quantity", \
            f"Header line incorrect: {header_line}"

        print(f"TASK 3 PASSED!")
        print(f"  Wrote {len(test_data)} rows to {test_file}")
        print(f"  Header: {header_line}")
        print(f"  First row: {lines[1].strip()}")

        # Test error handling
        bad_result = export_to_csv(test_data, test_headers, "/bad/path.csv")
        if bad_result is False:
            print("  Error handling works (bad path returned False)")
        else:
            print("  Warning: bad path should return False")
    except AssertionError as e:
        print(f"TASK 3 FAILED: {e}")
else:
    print("TASK 3: export_to_csv returned False")
    print("  Check that you can write files in this folder")

### Task 4: Bonus Challenge

Estimate total shipping cost by carrier mode (Ground, Air, Rail). This requires data from three tables and can be solved with SQL, Python, or a combination. Think about which approach makes the most sense.

In [ ]:
# YOUR CODE HERE (OPTIONAL) - Implement shipping cost by mode

def get_shipping_cost_by_mode(conn):
    """
    Estimate total shipping cost by carrier mode - BONUS.

    Calculate the estimated shipping cost for each carrier mode (Ground,
    Air, Rail). The cost estimate for each shipment is:

        quantity * unit_weight * base_rate

    This requires data from THREE tables: Shipment (quantity), Product
    (unit_weight), and Carrier (base_rate).

    Returns:
        list of tuples: (mode, estimated_cost) ordered by estimated_cost DESC
                        with costs rounded to 2 decimal places
    """
    pass

In [ ]:
# TEST - Task 4 (run this cell to check your work)

result = get_shipping_cost_by_mode(conn)

if result is None or len(result) == 0:
    print("Task 4: Not implemented (optional)")
else:
    try:
        assert len(result) == 3, f"Should have 3 modes, got {len(result)}"
        print("TASK 4 COMPLETED!")
        print("  Estimated shipping cost by mode:")
        for mode, cost in result:
            print(f"    {mode}: ${cost:,.2f}")
    except AssertionError as e:
        print(f"TASK 4 FAILED: {e}")

---

## Clean Up

In [ ]:
# Close the database connection when you're done
conn.close()
print("Database connection closed")